# Chapitre 8 – Architectures de réseaux de neurones profonds

Objectif : découvrir les **principales architectures** classiques en deep learning et comprendre la distinction entre **architectures** (blocs réutilisables) et **modèles** (assemblages pour une tâche précise).

Lien utile : [Neural Network Zoo](https://www.asimovinstitute.org/neural-network-zoo/)

## 1. Architectures vs Modèles

- **Architecture** = bloc réutilisable, brique de base (ex: bloc convolutif, bloc résiduel, bloc d'attention, Transformer encoder layer…)
- **Modèle** = assemblage organisé de plusieurs blocs pour résoudre une tâche précise (ex: ResNet-50, BERT, YOLOv8, EfficientNet…)

Avantage de la distinction : **modularité** et **réutilisabilité** du code.

## 2. Quelques architectures classiques

### 2.1 Multi-Layer Perceptron (MLP)

Réseau feed-forward entièrement connecté.

Usage fréquent :
- Données tabulaires
- Couche de tête (head) de classification / régression après un extracteur de caractéristiques

In [ ]:
import torch
import torch.nn as nn

class MLPBlock(nn.Module):
    def __init__(self, in_features, hidden_features, out_features):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        return self.fc2(x)

### 2.2 Convolutional Neural Network – Bloc convolutif

Brique de base des CNN : **Conv → BatchNorm → ReLU** (souvent MaxPool ou Stride)

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False)
        self.bn   = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))

### 2.3 Bloc résiduel (Residual Block)

Introduit par **ResNet** (2015)

Formule clé :  **y = F(x) + x**   (ou y = F(x) + projection(x) si changement de dimension)

Avantage principal : permet d'entraîner des réseaux **très profonds** sans disparition du gradient.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.relu  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)
        
    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity           # connexion skip
        return self.relu(out)

### 2.4 Mécanisme d’auto-attention (Transformer)

Cœur des architectures Transformer (Vaswani 2017).

Permet de pondérer dynamiquement l'importance des différentes parties de la séquence (ou de l'image dans ViT).

## 3. Quelques modèles emblématiques

### 3.1 LeNet-5 (1998)

- Premier CNN « moderne »
- Reconnaissance de chiffres manuscrits (MNIST)
- ~ 60 000 paramètres

### 3.2 VGG (2014)

- VGG16 / VGG19
- Très nombreuses convolutions 3×3 empilées
- ~ 138 M paramètres
- Images 224×224 RGB

### 3.3 ResNet (2015)

- ResNet-50, -101, -152…
- Blocs résiduels → réseaux très profonds
- Meilleure performance + moins de paramètres que VGG malgré plus de couches

## 4. Transfer Learning & Fine-tuning

Utiliser un modèle pré-entraîné (le plus souvent sur **ImageNet**) puis l’adapter à sa tâche.

Deux stratégies principales :
- **Feature extraction** : geler le backbone, entraîner seulement la tête
- **Fine-tuning** : dégeler progressivement certaines couches + faible learning rate

In [ ]:
import torchvision.models as models
from torchvision.models import VGG16_Weights

# Méthode recommandée (PyTorch ≥ 0.13)
model = models.vgg16(weights=VGG16_Weights.DEFAULT)

# Ou version plus ancienne / manuelle
# weights = torch.load('vgg16-397923af.pth')
# model.load_state_dict(weights)

## Exercice 1

1. Implémentez un **ResidualBlock** complet (avec projection si besoin)
2. Construisez un mini-ResNet (ex: ResNet-18 simplifié) en empilant plusieurs ResidualBlock
3. (Bonus) Adaptez la classification head pour 10 classes (CIFAR-10 par ex.)